> 在线讲义：[第 66 章 · VLN 的序列决策：POMDP 与信念状态、行为克隆的 O(T²) 与 DAgger 的 O(T)、语言条件策略、探索的信息增益](https://wikieden.github.io/math-and-eval-stats/lessons/66-vln-decision.html) · 本书《数学与实验统计》

# 第 66 章 · VLN 的序列决策：POMDP 与信念状态、行为克隆的 O(T²) 与 DAgger 的 O(T)、语言条件策略、探索的信息增益

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/66-vln-decision.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 信念更新 | 贝叶斯 | `bayes(prior, lik)` | 0.8889、0.9846 |
| 第二步 直方图滤波 | 预测 + 更新 | `corridor_step()` | 格 1、4 各 0.4211 |
| 第三步 误差界 | T²ε、uTε | `bc_bound`、`dagger_bound` | 10 vs 0.1（T = 100） |
| 第四步 FiLM | γ ⊙ h + β | `film()` | 两种指令两种输出 |
| 第五步 信息增益 | 熵的减少 | `info_gain()` | 0.4558 vs 0.5125 bit |

例子：两个房间的信念更新（看到冰箱之后在厨房的概率）；走廊上的直方图滤波（移动一步、再观测一次）；
行为克隆和 DAgger 的误差上界（只代入公式）；FiLM 式的语言条件；两个观测动作的信息增益。
里程碑 ME-M36 的“模拟行为克隆的误差累积 vs DAgger”留给自己做；这里只代入 DAgger 论文里的上界。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np

**函数 `bayes`**：b'(s) ∝ P(o | s) · b(s)。

In [ ]:
def bayes(prior, lik) -> np.ndarray:
    """b'(s) ∝ P(o | s) · b(s)。"""
    b = np.asarray(prior, float) * np.asarray(lik, float)
    return b / b.sum()

In [ ]:
DOORS = np.array([1, 0, 0, 1, 0])        # 走廊 5 格，哪些格有门

**函数 `corridor_step`**：预测：以 0.8 的概率真的走了 move 格（走廊是环形），0.2 原地不动。更新：门格看到门的概率 0.9，非门格 0.2。

In [ ]:
def corridor_step(belief, move: int = 1, p_ok: float = 0.8, saw_door: bool = True, p_hit: float = 0.9, p_false: float = 0.2):
    """预测：以 0.8 的概率真的走了 move 格（走廊是环形），0.2 原地不动。更新：门格看到门的概率 0.9，非门格 0.2。"""
    b = np.asarray(belief, float)
    pred = p_ok * np.roll(b, move) + (1 - p_ok) * b
    lik = np.where(DOORS == 1, p_hit, p_false) if saw_door else np.where(DOORS == 1, 1 - p_hit, 1 - p_false)
    return pred, bayes(pred, lik)

**函数 `bc_bound`**：行为克隆（DAgger 论文 §2）：每步犯错率 ε，T 步的总代价比专家多至多 T²ε（再大也不超过 T）。

In [ ]:
def bc_bound(eps: float, T: int) -> float:
    """行为克隆（DAgger 论文 §2）：每步犯错率 ε，T 步的总代价比专家多至多 T²ε（再大也不超过 T）。"""
    return min(T * T * eps, T)

**函数 `dagger_bound`**：DAgger（定理 3.2 的主项）：多出的代价 O(u·T·ε)，u 是一次犯错之后还要多付的代价。

In [ ]:
def dagger_bound(eps: float, T: int, u: float = 1.0) -> float:
    """DAgger（定理 3.2 的主项）：多出的代价 O(u·T·ε)，u 是一次犯错之后还要多付的代价。"""
    return min(u * T * eps, T)

**函数 `film`**：FiLM：γ = W_γ·l、β = W_β·l，输出 γ ⊙ h + β。同一视觉特征 h，不同指令得到不同的调制。

In [ ]:
def film(h, lang, Wg, Wb) -> np.ndarray:
    """FiLM：γ = W_γ·l、β = W_β·l，输出 γ ⊙ h + β。同一视觉特征 h，不同指令得到不同的调制。"""
    return (Wg @ lang) * h + Wb @ lang

In [ ]:
H = np.array([1.0, 0.5, -0.5])
WG = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
WB = np.array([[0.0, 0.5], [0.5, 0.0], [0.0, 0.0]])
LANG = {"左转": np.array([1.0, 0.0]), "右转": np.array([0.0, 1.0])}

**函数 `entropy`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def entropy(p) -> float:
    p = np.asarray(p, float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

**函数 `info_gain`**：IG = H(b) − Σ_o P(o) H(b | o)；lik_table[o][s] = P(o | s)。

In [ ]:
def info_gain(prior, lik_table) -> float:
    """IG = H(b) − Σ_o P(o) H(b | o)；lik_table[o][s] = P(o | s)。"""
    prior = np.asarray(prior, float)
    out = entropy(prior)
    for lik in lik_table:
        po = float(np.dot(lik, prior))
        if po > 0:
            out -= po * entropy(bayes(prior, lik))
    return out

In [ ]:
PRIOR4 = [0.4, 0.3, 0.2, 0.1]                     # 目标在 4 个房间里的信念
LOOK_LEFT = [[0.9, 0.9, 0.1, 0.1], [0.1, 0.1, 0.9, 0.9]]   # 看左边：能分清 {0, 1} 和 {2, 3}
LOOK_RIGHT = [[0.9, 0.1, 0.1, 0.1], [0.1, 0.9, 0.9, 0.9]]  # 看右边：只能认出房间 0

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 信念状态：两个房间")
b = bayes([0.5, 0.5], [0.8, 0.1])
print(f"  先验 [厨房 0.5, 卧室 0.5]，看到冰箱（P = 0.8 / 0.1）：后验 [{b[0]:.4f}, {b[1]:.4f}]")
b2 = bayes(b, [0.8, 0.1])
print(f"  再看到一次冰箱：[{b2[0]:.4f}, {b2[1]:.4f}]")

In [ ]:
print("\n第二步 走廊上的直方图滤波（门在格 0、3）")
b0 = np.full(5, 0.2)
_, b1 = corridor_step(b0, move=0, p_ok=1.0)
print(f"  均匀先验，原地看到门：{b1.round(4).tolist()}")
pred, b2 = corridor_step(b1, move=1)
print(f"  往右走一格（预测）：{pred.round(4).tolist()}")
_, b3 = corridor_step(b1, move=1, saw_door=False)
print(f"  走完没看到门：{b3.round(4).tolist()}")

In [ ]:
print("\n第三步 行为克隆 vs DAgger 的上界（只代入公式）")
for T in (10, 100, 1000):
    print(f"  ε = 0.001、T = {T}：BC ≤ {bc_bound(0.001, T):g}，DAgger ~ {dagger_bound(0.001, T):g}；一步都不错的概率 {(1 - 0.001) ** T:.4f}")

In [ ]:
print("\n第四步 语言条件策略：FiLM")
for name, l in LANG.items():
    print(f"  同一视觉特征 {H.tolist()}、指令“{name}”：{film(H, l, WG, WB).round(3).tolist()}")

In [ ]:
print("\n第五步 探索的信息增益")
print(f"  信念 {PRIOR4}：熵 {entropy(PRIOR4):.4f} bit")
for name, t in (("看左边", LOOK_LEFT), ("看右边", LOOK_RIGHT)):
    print(f"  {name}：信息增益 {info_gain(PRIOR4, t):.4f} bit")

## 3. 代码里才看得到的

1. **np.roll 假设走廊是环形的**：真实走廊有尽头，走到头的概率质量要留在原地，不能绕回另一头。
2. **观测不到也是观测**：“没看到门”同样要乘似然（1 − 0.9 或 1 − 0.2），只在看到东西时更新会让信念越来越偏。
3. **误差界只是代入公式**：模拟行为克隆的误差累积 vs DAgger、画出误差累积图，留给里程碑 ME-M36。

## 4. 自己改一改

1. 把 `corridor_step` 的门改到格 0 和格 2（不再对称），走一步、看一次之后信念还有两个峰吗？
2. 把 `LOOK_LEFT` 的正确率从 0.9 改成 0.99，它的信息增益能超过往右看吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看不见状态，信念当状态；先按动作推、再按观测乘；
> 走廊对称，峰会成双；
> BC 只学专家走过的路，一错就跑偏，T 方乘 ε；
> DAgger 让自己跑、专家来标，降成 T 乘 ε；
> 语言调制视觉，FiLM 给 γ、β；
> 信息增益是互信息，切得越对半看得越值。